In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib
import warnings
warnings.filterwarnings('ignore')

# Loading the full dataset (bank-full.csv from UCI)
df = pd.read_csv("bank-full.csv", sep=";")

print("Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nTarget distribution:")
print(df['y'].value_counts(normalize=True))

Shape: (45211, 17)

First 5 rows:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no



Column names:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']

Data types:
age           int64
job          object
marital      object
education    object
default      object
balance       int64
housing      object
loan         object
contact      object
day           int64
month        object
duration      int64
campaign      int64
pdays         int64
previous      int64
poutcome     object
y            object
dtype: object

Target distribution:
y
no     0.883015
yes    0.116985
Name: proportion, dtype: float64


In [ ]:
print("="*60)
print("DUPLICATES")
print("="*60)
print("Number of fully duplicated rows:", df.duplicated().sum())

In [ ]:
print("\n" + "="*60)
print("MISSING VALUES (NaN / None)")
print("="*60)
print(df.isnull().sum())

In [ ]:
print("\n" + "="*60)
print("'unknown' VALUES")
print("="*60)
unknown_counts = (df == "unknown").sum()
print(unknown_counts[unknown_counts > 0])

# Note: We keep "unknown" as a valid category.
# It is treated as meaningful information by OneHotEncoder.

In [ ]:
print("\n" + "="*60)
print("UNIQUE VALUES PER CATEGORICAL COLUMN")
print("="*60)
cat_cols = df.select_dtypes(include="object").columns
for col in cat_cols:
    print(f"\n{col}: {df[col].nunique()} unique values")
    print(df[col].value_counts())

In [ ]:
#Remove duration feature (known only AFTER the call → leakage)
df_clean = df.drop(columns=["duration"]).copy()

print("Shape after removing duration:", df_clean.shape)

In [ ]:
#Defining Feature Groups (after duration is removed)

# Numerical features (will be scaled)
num_features = ["age", "balance", "day", "campaign", "pdays", "previous"]

# Categorical features (will be one-hot encoded)
cat_features = ["job", "marital", "education", "default", 
                "housing", "loan", "contact", "month", "poutcome"]

print("Numerical features:", num_features)
print("Categorical features:", cat_features)
print("\nTotal features used:", len(num_features) + len(cat_features))

In [ ]:
#Encoding target: yes → 1, no → 0
df_clean["y"] = df_clean["y"].map({"yes": 1, "no": 0})

print("\nTarget distribution after encoding:")
print(df_clean["y"].value_counts(normalize=True))
print("\nColumns now:", df_clean.columns.tolist())
print("\nFirst 5 rows of uncleaned and cleaned data:")

print(df.head(5))
print(df_clean.head(5))

In [ ]:
from sklearn.model_selection import train_test_split

X = df_clean.drop(columns=["y"])
y = df_clean["y"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,          # keeps class balance the same in train & test
    random_state=42      # fixed seed for reproducibility
)

print("Train shape:", X_train.shape)
print("Test  shape:", X_test.shape)
print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True).round(4))
print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).round(4))

In [ ]:
# Numeric transformer
numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

# Categorical transformer
categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Combining them
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_features),
        ("cat", categorical_transformer, cat_features)
    ],
    remainder="drop"   # drop any columns not listed (safety)
)

print("Preprocessor created successfully.")
print(preprocessor)

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print("Processed train shape:", X_train_processed.shape)
print("Processed test  shape:", X_test_processed.shape)

# Quick check – first 5 rows of processed train data
feature_names = (
    num_features +
    list(preprocessor.named_transformers_["cat"]
         .named_steps["onehot"]
         .get_feature_names_out(cat_features))
)

X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)
display(X_train_df.head())

In [ ]:
joblib.dump(preprocessor, "ds52-preprocessor.joblib")
print("Saved: ds52-preprocessor.joblib")

joblib.dump((X_train, X_test, y_train, y_test), "ds52-train_test_split.joblib")
print("Saved: ds52-train_test_split.joblib")